# Week 4: Niagara Census Self-Employment Data Cleaning

## 1. Introduction

This notebook documents the cleaning and validation of the 2021 Statistics Canada Census Class of Worker and Self-Employment dataset for Niagara Census Division.

**Data source:** Statistics Canada, Table 98-10-0592-01

**Objectives:**
- Load and inspect the raw Census dataset.
- Validate data quality and consistency.
- Preserve important age, gender, industry, and class-of-worker categories.
- Prepare a cleaned dataset for further analysis.
- Document cleaning decisions and limitations.

**Important:** The original raw dataset will remain unchanged.

In [1]:
# Import the library for working with datasets
import pandas as pd

print("Pandas imported successfully!")

Pandas imported successfully!


In [2]:
# Load the 2021 Niagara Census dataset
file_path = "../../data/raw/9810059201_databaseLoadingData.csv"

df = pd.read_csv(file_path, low_memory=False)

print("Dataset shape:", df.shape)

Dataset shape: (125714, 20)


In [3]:
# Display the first five rows
df.head()

,REF_DATE,GEO,DGUID,Statistics (3),Labour force status (3),Age (15A),Gender (3),Industry - Groups - North American Industry Classification System (NAICS) 2017 (428A),Class of worker (7A),UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2021,Niagara,2021A00033526,Count,Total - Labour force status,Total - Age,Total - Gender,Total - Industry - Groups - North American Ind...,Total - Class of worker,NaN,0,units,0,NaN,173.1.1.1.1.1.1,233555,NaN,NaN,NaN,0
1,2021,Niagara,2021A00033526,Count,Total - Labour force status,Total - Age,Total - Gender,Total - Industry - Groups - North American Ind...,Class of worker - not applicable,NaN,0,units,0,NaN,173.1.1.1.1.1.2,6460,NaN,NaN,NaN,0
2,2021,Niagara,2021A00033526,Count,Total - Labour force status,Total - Age,Total - Gender,Total - Industry - Groups - North American Ind...,All classes of workers,NaN,0,units,0,NaN,173.1.1.1.1.1.3,227095,NaN,NaN,NaN,0
3,2021,Niagara,2021A00033526,Count,Total - Labour force status,Total - Age,Total - Gender,Total - Industry - Groups - North American Ind...,Employee,NaN,0,units,0,NaN,173.1.1.1.1.1.4,194420,NaN,NaN,NaN,0
4,2021,Niagara,2021A00033526,Count,Total - Labour force status,Total - Age,Total - Gender,Total - Industry - Groups - North American Ind...,Self-employed,NaN,0,units,0,NaN,173.1.1.1.1.1.5,32680,NaN,NaN,NaN,0


In [4]:
# Check column names, data types, and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 125714 entries, 0 to 125713
Data columns (total 20 columns):
 #   Column                                                                                 Non-Null Count   Dtype  
---  ------                                                                                 --------------   -----  
 0   REF_DATE                                                                               125714 non-null  int64  
 1   GEO                                                                                    125714 non-null  str    
 2   DGUID                                                                                  125714 non-null  str    
 3   Statistics (3)                                                                         125714 non-null  str    
 4   Labour force status (3)                                                                125714 non-null  str    
 5   Age (15A)                                                                   

In [5]:
# Create a working copy for cleaning
clean_df = df.copy()

# Confirm the copy has the same dimensions
print("Original dataset:", df.shape)
print("Working copy:", clean_df.shape)

Original dataset: (125714, 20)
Working copy: (125714, 20)


In [6]:
# Remove columns that are completely empty
empty_columns = [
    "UOM",
    "VECTOR",
    "STATUS",
    "SYMBOL",
    "TERMINATED"
]

clean_df = clean_df.drop(columns=empty_columns)

# Check the new dimensions
print("Before cleaning:", df.shape)
print("After removing empty columns:", clean_df.shape)

Before cleaning: (125714, 20)
After removing empty columns: (125714, 15)


In [7]:
# Count missing values in each remaining column
missing_values = clean_df.isnull().sum()

print(missing_values)

REF_DATE                                                                                 0
GEO                                                                                      0
DGUID                                                                                    0
Statistics (3)                                                                           0
Labour force status (3)                                                                  0
Age (15A)                                                                                0
Gender (3)                                                                               0
Industry - Groups - North American Industry Classification System (NAICS) 2017 (428A)    0
Class of worker (7A)                                                                     0
UOM_ID                                                                                   0
SCALAR_FACTOR                                                                            0

In [8]:
# Count rows that are exact duplicates
duplicate_count = clean_df.duplicated().sum()

print("Duplicate records:", duplicate_count)

Duplicate records: 0


In [9]:
# Check for duplicate census coordinates
duplicate_coordinates = clean_df["COORDINATE"].duplicated().sum()

print("Duplicate coordinates:", duplicate_coordinates)

Duplicate coordinates: 0


In [10]:
# Extract the class-of-worker ID from each census coordinate
clean_df["class_worker_id"] = (
    clean_df["COORDINATE"].str.split(".").str[-1].astype(int)
)

# Display the IDs used for Self-employed
clean_df.loc[
    clean_df["Class of worker (7A)"] == "Self-employed",
    ["class_worker_id", "Class of worker (7A)"]
].drop_duplicates().sort_values("class_worker_id")

,class_worker_id,Class of worker (7A)
4,5,Self-employed
5,6,Self-employed


In [11]:
# Create clearer class-of-worker labels
clean_df["class_worker_label"] = clean_df["Class of worker (7A)"]

# Distinguish the two self-employed categories
clean_df.loc[
    clean_df["class_worker_id"] == 5,
    "class_worker_label"
] = "Self-employed (including unpaid family workers)"

clean_df.loc[
    clean_df["class_worker_id"] == 6,
    "class_worker_label"
] = "Self-employed (excluding unpaid family workers)"

# Check the results
clean_df[
    ["class_worker_id", "class_worker_label"]
].drop_duplicates().sort_values("class_worker_id")

,class_worker_id,class_worker_label
0,1,Total - Class of worker
1,2,Class of worker - not applicable
2,3,All classes of workers
3,4,Employee
4,5,Self-employed (including unpaid family workers)
5,6,Self-employed (excluding unpaid family workers)
6,7,Unpaid family worker


In [12]:
# Extract the industry ID from the census coordinate
clean_df["industry_id"] = (
    clean_df["COORDINATE"].str.split(".").str[5].astype(int)
)

# Preview the industry identifiers and names
clean_df[
    [
        "industry_id",
        "Industry - Groups - North American Industry Classification System (NAICS) 2017 (428A)"
    ]
].drop_duplicates().head(10)

,industry_id,Industry - Groups - North American Industry Classification System (NAICS) 2017 (428A)
0,1,Total - Industry - Groups - North American Ind...
7,2,Industry - not applicable
9,3,All industries
15,4,"11 Agriculture, forestry, fishing and hunting"
21,5,111-112 Farms
27,6,"1110 Farms (except greenhouses, nurseries, flo..."
33,7,"1114 Greenhouse, nursery and floriculture prod..."
38,9,113 Forestry and logging
43,12,1133 Logging
48,13,"114 Fishing, hunting and trapping"


In [13]:
# Check whether any industry ID has multiple names
industry_check = clean_df.groupby("industry_id")[
    "Industry - Groups - North American Industry Classification System (NAICS) 2017 (428A)"
].nunique()

print("Industry IDs with multiple names:", (industry_check > 1).sum())

Industry IDs with multiple names: 0


In [14]:
# Extract the age category ID
clean_df["age_id"] = (
    clean_df["COORDINATE"].str.split(".").str[3].astype(int)
)

# Display the age categories and their IDs
clean_df[
    ["age_id", "Age (15A)"]
].drop_duplicates().sort_values("age_id")

,age_id,Age (15A)
0,1,Total - Age
5130,2,15 to 24 years
8053,3,15 to 19 years
9802,4,20 to 24 years
12538,5,25 to 64 years
17484,6,25 to 54 years
22174,7,25 to 34 years
25820,8,25 to 29 years
28918,9,30 to 34 years
32088,10,35 to 44 years


In [15]:
# Extract the gender category ID
clean_df["gender_id"] = (
    clean_df["COORDINATE"].str.split(".").str[4].astype(int)
)

# Display unique gender categories and their IDs
clean_df[
    ["gender_id", "Gender (3)"]
].drop_duplicates().sort_values("gender_id")

,gender_id,Gender (3)
0,1,Total - Gender
1831,2,Men+
3565,3,Women+


In [16]:
# Extract the labour force status ID
clean_df["labour_status_id"] = (
    clean_df["COORDINATE"].str.split(".").str[2].astype(int)
)

# Display unique labour force categories and their IDs
clean_df[
    ["labour_status_id", "Labour force status (3)"]
].drop_duplicates().sort_values("labour_status_id")

,labour_status_id,Labour force status (3)
0,1,Total - Labour force status
51663,2,Employed
101329,3,Unemployed


In [17]:
# Check the number of unique IDs in each category
print("Age categories:", clean_df["age_id"].nunique())
print("Gender categories:", clean_df["gender_id"].nunique())
print("Labour force categories:", clean_df["labour_status_id"].nunique())
print("Industry categories:", clean_df["industry_id"].nunique())
print("Class of worker categories:", clean_df["class_worker_id"].nunique())

Age categories: 15
Gender categories: 3
Labour force categories: 3
Industry categories: 400
Class of worker categories: 7


In [18]:
# Compare the original and cleaned datasets
print("Original dataset:", df.shape)
print("Cleaned dataset:", clean_df.shape)

# Check whether any records were lost
print("Records lost:", len(df) - len(clean_df))

Original dataset: (125714, 20)
Cleaned dataset: (125714, 21)
Records lost: 0


In [19]:
# Check missing values in the working dataset
print("Total missing values:", clean_df.isna().sum().sum())

Total missing values: 0


In [20]:
# Compare original and cleaned census counts
values_unchanged = df["VALUE"].equals(clean_df["VALUE"])

print("Census values unchanged:", values_unchanged)

Census values unchanged: True


In [21]:
# Confirm all original census coordinates are preserved
coordinates_unchanged = df["COORDINATE"].equals(clean_df["COORDINATE"])

print("Census coordinates unchanged:", coordinates_unchanged)

Census coordinates unchanged: True


In [22]:
# Confirm the cleaned data contains only Niagara Census Division
clean_df.groupby(["GEO", "DGUID"]).size()

GEO      DGUID        
Niagara  2021A00033526    125714
dtype: int64

In [23]:
# Save the cleaned Census dataset
output_path = "../../data/interim/census_self_employment_niagara_cleaned.csv"

clean_df.to_csv(output_path, index=False)

print("Cleaned dataset exported successfully!")
print("File location:", output_path)
print("Dataset shape:", clean_df.shape)

Cleaned dataset exported successfully!
File location: ../../data/interim/census_self_employment_niagara_cleaned.csv
Dataset shape: (125714, 21)


In [24]:
# Read the exported CSV to verify it
verified_df = pd.read_csv(output_path, low_memory=False)

print("Exported dataset shape:", verified_df.shape)
print("Values preserved:", verified_df["VALUE"].equals(clean_df["VALUE"]))
print("Coordinates preserved:", verified_df["COORDINATE"].equals(clean_df["COORDINATE"]))

Exported dataset shape: (125714, 21)
Values preserved: True
Coordinates preserved: True
